# 🏗️ Building Agent — Autonomous Building Detection & Data Collection

Zero-cost pipeline for detecting, classifying, and auditing buildings from aerial imagery.

## Features
- **Detection**: YOLOv8n-seg segmentation → building masks
- **Classification**: Size, shape, color, roof type, building type
- **Addresses**: OSM addr tags → Nominatim reverse geocode fallback
- **Audit**: Spatial comparison with OSM records → flag unrecorded buildings
- **Self-improvement**: Export training data → fine-tune model → better detection
- **Change detection**: Re-scan to find new/demolished buildings
- **Export**: Excel, GeoJSON, Google Sheets

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Add project root to path if running from notebook
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from agent.config import load_config
from agent.store import BuildingStore
from agent.pipeline import BuildingPipeline
from agent.export import export_excel, export_geojson, export_google_sheets

print('[OK] Agent modules loaded')

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────
config = load_config()
print(f'AOI bbox: {config.area.bbox_wgs84}')
print(f'Model: {config.model.name}')
print(f'Min building area: {config.classification.min_area_m2} m²')

In [ ]:
# ── Initialize Store & Pipeline ────────────────────────────────────────────
store = BuildingStore(config.storage.database_path)
pipeline = BuildingPipeline(config, store)
print(f'Database: {config.storage.database_path}')

In [ ]:
# ── Run Scan ───────────────────────────────────────────────────────────────
# ⚠️ This downloads imagery and runs YOLO — may take 5-30 minutes on Colab
scan_id = pipeline.run_scan(
    bbox=config.area.bbox_wgs84,
)
print(f'Scan complete: {scan_id}')

In [ ]:
# ── View Results ───────────────────────────────────────────────────────────
buildings = store.get_buildings_by_scan(scan_id)
print(f'Total buildings: {len(buildings)}')
print(f'Unrecorded: {sum(1 for b in buildings if b.is_unrecorded)}')

# Show first 5 buildings
for b in buildings[:5]:
    print(f'  {b.building_id[:16]} | '
          f'lat={b.latitude:.4f} lon={b.longitude:.4f} | '
          f'{b.area_m2:.0f}m² | '
          f'{b.size_class.value}/{b.roof_type.value} | '
          f'{"UNRECORDED" if b.is_unrecorded else "recorded"}')

In [ ]:
# ── Export Results ──────────────────────────────────────────────────────────
export_dir = config.storage.export_dir

excel_path = f'{export_dir}/buildings_{scan_id[:12]}.xlsx'
export_excel(store, excel_path, scan_id)
print(f'Excel: {excel_path}')

geojson_path = f'{export_dir}/buildings_{scan_id[:12]}.geojson'
export_geojson(store, geojson_path, scan_id)
print(f'GeoJSON: {geojson_path}')

In [ ]:
# ── Export Training Data (Self-Improvement Loop) ───────────────────────────
from agent.training import export_yolo_labels

dataset_yaml = export_yolo_labels(store, scan_id=scan_id)
if dataset_yaml:
    print(f'Training data: {dataset_yaml}')
    print()
    print('To fine-tune the model, run:')
    print(f'  yolo segment train model={config.model.name} '
          f'data={dataset_yaml} epochs=100 imgsz=640')

In [ ]:
# ── Interactive Map ─────────────────────────────────────────────────────────
import folium
import json

bbox = config.area.bbox_wgs84
m = folium.Map(
    location=[(bbox[1] + bbox[3]) / 2, (bbox[0] + bbox[2]) / 2],
    zoom_start=17,
    tiles='OpenStreetMap',
)

for b in buildings:
    color = 'red' if b.is_unrecorded else 'green'
    folium.CircleMarker(
        location=[b.latitude, b.longitude],
        radius=5,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=f'{b.building_id[:12]}: {b.area_m2:.0f}m² conf={b.confidence:.2f}',
    ).add_to(m)

display(m)

In [ ]:
# ── Change Detection (after 2+ scans) ──────────────────────────────────────
scans = store.get_all_scans()
if len(scans) >= 2:
    changes = store.detect_changes(scans[0].scan_id, scans[1].scan_id)
    print(f'New: {len(changes["new"])}')
    print(f'Unchanged: {len(changes["unchanged"])}')
    print(f'Demolished: {len(changes["demolished"])}')
else:
    print('Need at least 2 scans for change detection. Run another scan.')

In [ ]:
# ── Summary ─────────────────────────────────────────────────────────────────
print('=' * 50)
print('  BUILDING AGENT — SCAN SUMMARY')
print('=' * 50)
print(f'  AOI: {config.area.bbox_wgs84}')
print(f'  Total buildings: {len(buildings)}')
print(f'  Unrecorded: {sum(1 for b in buildings if b.is_unrecorded)}')
print(f'  With addresses: {sum(1 for b in buildings if b.house_number)}')
print()
print(f'  Type distribution:')
from collections import Counter
type_counts = Counter(b.building_type.value for b in buildings)
for t, c in type_counts.most_common():
    print(f'    {t}: {c}')
print()
print(f'  Roof distribution:')
roof_counts = Counter(b.roof_type.value for b in buildings)
for r, c in roof_counts.most_common():
    print(f'    {r}: {c}')
print('=' * 50)